**Recommended Colab workflow (Google Drive).** Sync this repository from the MacBook to `MyDrive/NRC_CALIB_CODE` (for example using the SSH/rsync cell), open this notebook in Colab, then run the cells in order. A Colab runtime cannot directly mount an arbitrary MacBook folder: Drive or a reachable SSH endpoint is required.


## Execution contract

Every code cell is deliberately self-contained: it re-discovers or creates `PROJECT_ROOT` from the `NRC_CAL_PROJECT_ROOT` environment variable, then sensible Drive and Colab defaults. Cells may therefore be rerun independently after a runtime restart. The default is Drive-first, and all persistent outputs are written beneath the selected project root.

The required project skeleton is created if it does not exist:

```text
NRC_CALIB_CODE/{notebooks,src/{datasets,models,metrics,geometry,calibration,plotting,utils},configs,outputs,figures,checkpoints,logs,tests,external}
```

Running a synchronization cell never uses `--delete`; it only adds or updates destination files. Inspect and set the configuration variables in the relevant cell before using SSH.

In [1]:
# Cell 1 - Drive-first project-root discovery (safe to run independently).
from __future__ import annotations

import os
from pathlib import Path
from typing import Iterable

PROJECT_NAME = "NRC_CALIB_CODE"
PROJECT_DIRNAME = PROJECT_NAME
REQUIRED_DIRECTORIES = (
    "notebooks", "src", "src/datasets", "src/models", "src/metrics", "src/geometry",
    "src/calibration", "src/plotting", "src/utils", "configs", "outputs", "figures",
    "checkpoints", "logs", "tests", "external",
)

def _is_colab() -> bool:
    """Return whether this interpreter is running inside Google Colab."""
    try:
        import google.colab  # type: ignore  # noqa: F401
        return True
    except ImportError:
        return False

def _mount_drive_if_available() -> Path | None:
    """Mount Drive in Colab and return its mount point, or None elsewhere."""
    if not _is_colab():
        return None
    from google.colab import drive  # type: ignore
    mount_point = Path("/content/drive")
    try:
        drive.mount(str(mount_point), force_remount=False)
    except Exception as error:
        raise RuntimeError("Google Drive mounting failed; authenticate in the Colab prompt.") from error
    return mount_point

def _candidate_roots(drive_root: Path | None) -> Iterable[Path]:
    """Yield roots in priority order without scanning the whole Drive."""
    explicit = os.environ.get("NRC_CAL_PROJECT_ROOT")
    if explicit:
        yield Path(explicit).expanduser()
    if drive_root is not None:
        yield drive_root / "MyDrive" / PROJECT_DIRNAME
    yield Path("/content") / PROJECT_DIRNAME
    yield Path.cwd() / PROJECT_DIRNAME

drive_root = _mount_drive_if_available()
candidates = list(_candidate_roots(drive_root))
existing = next((path for path in candidates if path.is_dir()), None)
PROJECT_ROOT = existing or candidates[0]
for relative_dir in REQUIRED_DIRECTORIES:
    (PROJECT_ROOT / relative_dir).mkdir(parents=True, exist_ok=True)
project_is_on_drive = drive_root is not None and str(PROJECT_ROOT.resolve()).startswith(str(drive_root.resolve()))
default_checkpoint_root = PROJECT_ROOT / "checkpoints"
CHECKPOINT_ROOT = Path(os.environ.get("NRC_CAL_CHECKPOINT_ROOT", str(default_checkpoint_root))).expanduser()
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
os.environ["NRC_CAL_PROJECT_ROOT"] = str(PROJECT_ROOT.resolve())
os.environ["NRC_CAL_CHECKPOINT_ROOT"] = str(CHECKPOINT_ROOT.resolve())

print(f"Colab runtime: {_is_colab()}")
print(f"Google Drive: {drive_root if drive_root else 'not available'}")
print(f"PROJECT_ROOT: {PROJECT_ROOT.resolve()}")
print(f"CHECKPOINT_ROOT: {CHECKPOINT_ROOT.resolve()}")
print("Project directories verified: " + ", ".join(REQUIRED_DIRECTORIES))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Colab runtime: True
Google Drive: /content/drive
PROJECT_ROOT: /content/drive/MyDrive/NRC_CALIB_CODE
CHECKPOINT_ROOT: /content/drive/MyDrive/NRC_CALIB_CODE/checkpoints
Project directories verified: notebooks, src, src/datasets, src/models, src/metrics, src/geometry, src/calibration, src/plotting, src/utils, configs, outputs, figures, checkpoints, logs, tests, external


## Install the pinned research environment

This cell installs the requested stack only when a package is absent. Colab's CUDA-enabled PyTorch build is preserved when it is already present; installing another PyTorch wheel unnecessarily can remove CUDA support. The generated `configs/requirements-colab.txt` records the declared package requirements, while the final cell records the exact resolved versions.

**Estimated resources:** 2--7 minutes and no GPU allocation. **Expected output:** one `present` or `installing` line per dependency, followed by a successful import check.

In [2]:
# Cell 2 - Idempotent dependency installation (safe to run independently).
from __future__ import annotations

import importlib.util
import os
import subprocess
import sys
from pathlib import Path

PROJECT_ROOT = Path(os.environ.get("NRC_CAL_PROJECT_ROOT", "/content/NRC_CALIB_CODE")).expanduser()
for relative_dir in ("configs", "outputs", "logs"):
    (PROJECT_ROOT / relative_dir).mkdir(parents=True, exist_ok=True)

# Distribution name, import name, and conservative Colab-compatible requirement.
PACKAGES: tuple[tuple[str, str, str], ...] = (
    ("torch", "torch", "torch>=2.2"),
    ("lightning", "lightning", "lightning>=2.2"),
    ("numpy", "numpy", "numpy>=1.26"),
    ("scipy", "scipy", "scipy>=1.11"),
    ("pandas", "pandas", "pandas>=2.1"),
    ("matplotlib", "matplotlib", "matplotlib>=3.8"),
    ("plotly", "plotly", "plotly>=5.18"),
    ("seaborn", "seaborn", "seaborn>=0.13"),
    ("scikit-learn", "sklearn", "scikit-learn>=1.4"),
    ("hydra-core", "hydra", "hydra-core>=1.3"),
    ("tqdm", "tqdm", "tqdm>=4.66"),
    ("einops", "einops", "einops>=0.7"),
    ("PyYAML", "yaml", "PyYAML>=6.0"),
    ("rich", "rich", "rich>=13.7"),
    ("jupyterlab", "jupyterlab", "jupyterlab>=4.0"),
    ("wandb", "wandb", "wandb>=0.16"),
    ("umap-learn", "umap", "umap-learn>=0.5"),
    ("psutil", "psutil", "psutil>=5.9"),
    ("pytest", "pytest", "pytest>=8.0"),
)

def ensure_package(distribution: str, import_name: str, requirement: str) -> None:
    """Install a requirement only if its Python import is unavailable."""
    if importlib.util.find_spec(import_name) is not None:
        print(f"present: {distribution}")
        return
    print(f"installing: {requirement}")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", requirement],
        check=True,
    )

for distribution, import_name, requirement in PACKAGES:
    ensure_package(distribution, import_name, requirement)

requirements_path = PROJECT_ROOT / "configs" / "requirements-colab.txt"
requirements_path.write_text(
    "\n".join(requirement for _, _, requirement in PACKAGES) + "\n",
    encoding="utf-8",
)
for _, import_name, _ in PACKAGES:
    __import__(import_name)
print(f"Dependency imports verified. Requirements declaration: {requirements_path}")


present: torch
present: lightning
present: numpy
present: scipy
present: pandas
present: matplotlib
present: plotly
present: seaborn
present: scikit-learn
present: hydra-core
present: tqdm
present: einops
present: PyYAML
present: rich
present: jupyterlab
present: wandb
present: umap-learn
present: psutil
present: pytest
Dependency imports verified. Requirements declaration: /content/drive/MyDrive/NRC_CALIB_CODE/configs/requirements-colab.txt


## Project synchronization options

Use exactly one of the next two cells per runtime. Google Drive is the default and requires no public network access to the MacBook after the project is uploaded or synchronized to Drive. SSH/rsync is useful when the MacBook is reachable from Colab (for example through a secured tunnel); it deliberately requires explicit environment variables and never prints credentials.


In [3]:
# Cell 3A - Preferred option: select a project already synchronized to Google Drive.
from __future__ import annotations

import os
from pathlib import Path

try:
    from google.colab import drive  # type: ignore
except ImportError as error:
    raise RuntimeError("Drive synchronization is available only in Google Colab.") from error

drive.mount("/content/drive", force_remount=False)
# Override this variable before execution if the Drive layout differs.
DRIVE_PROJECT_ROOT = Path(os.environ.get("NRC_CAL_DRIVE_PROJECT_ROOT", "/content/drive/MyDrive/NRC_CALIB_CODE"))
if not DRIVE_PROJECT_ROOT.is_dir():
    raise FileNotFoundError(
        f"Drive project not found at {DRIVE_PROJECT_ROOT}. Sync the MacBook project there, "
        "or set NRC_CAL_DRIVE_PROJECT_ROOT to its actual location."
    )
for relative_dir in ("notebooks", "src", "src/datasets", "src/models", "src/metrics", "src/geometry", "src/calibration", "src/plotting", "src/utils", "configs", "outputs", "figures", "checkpoints", "logs", "tests", "external"):
    (DRIVE_PROJECT_ROOT / relative_dir).mkdir(parents=True, exist_ok=True)
os.environ["NRC_CAL_PROJECT_ROOT"] = str(DRIVE_PROJECT_ROOT.resolve())
print(f"Using Drive project directly: {DRIVE_PROJECT_ROOT.resolve()}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using Drive project directly: /content/drive/MyDrive/NRC_CALIB_CODE


In [4]:
# Cell 3B - SSH/rsync option: pull the project from a reachable MacBook without deleting files.
from __future__ import annotations

import os
import shutil
import subprocess
from pathlib import Path

PROJECT_ROOT = Path(os.environ.get("NRC_CAL_PROJECT_ROOT", "/content/NRC_CALIB_CODE")).expanduser()
SSH_HOST = os.environ.get("NRC_CAL_MAC_SSH_HOST", "MacBook-Pro-cua-Hac.local").strip()
SSH_USER = os.environ.get("NRC_CAL_MAC_SSH_USER", "hacminhquan").strip()
SSH_REMOTE_PATH = os.environ.get("NRC_CAL_MAC_PROJECT_PATH", "/Users/hacminhquan/Documents/MAIN/HCMUT/0.URA/7.Viettel Calibration/NEW ERA/NRC-Cal/NRC_Calib_Code").strip()
SSH_PORT = os.environ.get("NRC_CAL_MAC_SSH_PORT", "22").strip()
if not all((SSH_HOST, SSH_USER, SSH_REMOTE_PATH)):
    raise ValueError(
        "Set NRC_CAL_MAC_SSH_HOST, NRC_CAL_MAC_SSH_USER, and NRC_CAL_MAC_PROJECT_PATH. "
        "Use a key loaded into the runtime; secrets are not accepted in notebook source."
    )
if shutil.which("rsync") is None:
    raise RuntimeError("rsync is unavailable in this runtime. Install it with apt before using this option.")

PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
remote = f"{SSH_USER}@{SSH_HOST}:{SSH_REMOTE_PATH.rstrip('/')}/"
ssh_command = f"ssh -p {SSH_PORT} -o StrictHostKeyChecking=accept-new"
command = [
    "rsync", "-az", "-s", "--partial", "--exclude", ".git", "-e", ssh_command,
    remote, str(PROJECT_ROOT) + "/",
]
subprocess.run(command, check=True)
os.environ["NRC_CAL_PROJECT_ROOT"] = str(PROJECT_ROOT.resolve())
print(f"Synchronized project into: {PROJECT_ROOT.resolve()}")


ValueError: Set NRC_CAL_MAC_SSH_HOST, NRC_CAL_MAC_SSH_USER, and NRC_CAL_MAC_PROJECT_PATH. Use a key loaded into the runtime; secrets are not accepted in notebook source.

## Clone and inspect upstream baselines

The two cited repositories are cloned under `external/` if missing. This cell does not claim that checkpoints are available merely because a repository cloned: it recursively inventories recognized checkpoint suffixes, records SHA-256 hashes and sizes, and reports an explicit failure if no checkpoints are found. Some upstream projects require a separate dataset/checkpoint download step, which will be handled in `01_download_repositories.ipynb` after their repository documentation is inspected.

**Estimated resources:** 1--3 minutes plus repository size; no GPU memory. **Expected output:** repository commit IDs and a checkpoint inventory JSON.

In [ ]:
# Cell 4 - Clone upstream repositories and produce an auditable checkpoint inventory.
from __future__ import annotations

import hashlib
import json
import os
import subprocess
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

PROJECT_ROOT = Path(os.environ.get("NRC_CAL_PROJECT_ROOT", "/content/NRC_CALIB_CODE")).expanduser()
EXTERNAL_ROOT = PROJECT_ROOT / "external"
EXTERNAL_ROOT.mkdir(parents=True, exist_ok=True)
UPSTREAM_REPOSITORIES = {
    "quantile-recalibration-training": "https://github.com/Vekteur/quantile-recalibration-training.git",
    "probabilistic-calibration-study": "https://github.com/Vekteur/probabilistic-calibration-study.git",
}
CHECKPOINT_SUFFIXES = {".ckpt", ".pt", ".pth", ".pkl", ".joblib", ".safetensors"}

def git_output(repository: Path, *args: str) -> str:
    """Return trimmed git output or raise with command context."""
    completed = subprocess.run(
        ["git", "-C", str(repository), *args], check=True, text=True, capture_output=True
    )
    return completed.stdout.strip()

def sha256(path: Path, chunk_size: int = 1 << 20) -> str:
    """Hash a checkpoint incrementally so large files do not exhaust RAM."""
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

inventory: dict[str, Any] = {"created_utc": datetime.now(timezone.utc).isoformat(), "repositories": {}}
for name, url in UPSTREAM_REPOSITORIES.items():
    destination = EXTERNAL_ROOT / name
    if not destination.exists():
        subprocess.run(["git", "clone", "--depth", "1", url, str(destination)], check=True)
    if not (destination / ".git").is_dir():
        raise RuntimeError(f"Expected a git repository at {destination}, found a non-git directory.")
    checkpoints = []
    for path in sorted(destination.rglob("*")):
        if path.is_file() and path.suffix.lower() in CHECKPOINT_SUFFIXES:
            checkpoints.append({
                "path": str(path.relative_to(PROJECT_ROOT)),
                "bytes": path.stat().st_size,
                "sha256": sha256(path),
            })
    inventory["repositories"][name] = {
        "url": git_output(destination, "config", "--get", "remote.origin.url"),
        "commit": git_output(destination, "rev-parse", "HEAD"),
        "checkpoints": checkpoints,
    }
    print(f"{name}: {len(checkpoints)} checkpoint file(s) at {inventory['repositories'][name]['commit']}")

inventory_path = PROJECT_ROOT / "outputs" / "upstream_checkpoint_inventory.json"
inventory_path.parent.mkdir(parents=True, exist_ok=True)
inventory_path.write_text(json.dumps(inventory, indent=2, sort_keys=True), encoding="utf-8")
missing = [name for name, details in inventory["repositories"].items() if not details["checkpoints"]]
print(f"Checkpoint inventory saved: {inventory_path}")
if missing:
    print("CHECKPOINT VERIFICATION INCOMPLETE: no checkpoint files found in " + ", ".join(missing))
else:
    print("CHECKPOINT VERIFICATION PASSED: every discovered checkpoint has a SHA-256 hash.")


quantile-recalibration-training: 0 checkpoint file(s) at 9dafa605f14f52b99c898b20b26f97ede9f99074
probabilistic-calibration-study: 0 checkpoint file(s) at 22827a04ca33755a0686b568727e0262845e33eb
Checkpoint inventory saved: /content/drive/MyDrive/NRC_CALIB_CODE/outputs/upstream_checkpoint_inventory.json
CHECKPOINT VERIFICATION INCOMPLETE: no checkpoint files found in quantile-recalibration-training, probabilistic-calibration-study


## GPU, CUDA, and PyTorch validation

This check reports the CUDA driver through `nvidia-smi`, the CUDA runtime linked to PyTorch, available GPU memory, CPU RAM, and PyTorch's ability to execute a CUDA tensor operation. It enables TensorFloat-32 where supported and provides a mixed-precision context factory for later notebooks. No model weights are allocated.

**Estimated resources:** under 30 seconds; less than 100 MB GPU memory. **Expected output:** `CUDA usable: True` on a GPU Colab runtime. If false, switch Colab to a GPU runtime before proceeding.

In [ ]:
# Cell 5 - Hardware validation and mixed-precision configuration (safe to run independently).
from __future__ import annotations

import os
import platform
import shutil
import subprocess
from contextlib import nullcontext
from pathlib import Path
from typing import ContextManager

import psutil
import torch

PROJECT_ROOT = Path(os.environ.get("NRC_CAL_PROJECT_ROOT", "/content/NRC_CALIB_CODE")).expanduser()
(PROJECT_ROOT / "logs").mkdir(parents=True, exist_ok=True)
CUDA_USABLE = torch.cuda.is_available()
if CUDA_USABLE:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.set_float32_matmul_precision("high")
    # This operation verifies that the installed PyTorch binary can actually execute on CUDA.
    probe = (torch.ones(1, device="cuda") + 1).item()
    assert probe == 2.0

def mixed_precision_context() -> ContextManager[object]:
    """Return CUDA bfloat16 autocast when available, otherwise a no-op context."""
    if not torch.cuda.is_available():
        return nullcontext()
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    return torch.autocast(device_type="cuda", dtype=dtype)

print(f"Python: {platform.python_version()}")
print(f"PyTorch: {torch.__version__}")
print(f"PyTorch CUDA runtime: {torch.version.cuda}")
print(f"CUDA usable: {CUDA_USABLE}")
print(f"Host RAM: {psutil.virtual_memory().total / 2**30:.2f} GiB")
if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"], text=True, capture_output=True, check=False).stdout.strip())
else:
    print("nvidia-smi unavailable")
if CUDA_USABLE:
    properties = torch.cuda.get_device_properties(0)
    print(f"GPU: {properties.name}; memory: {properties.total_memory / 2**30:.2f} GiB")
    print(f"Mixed precision dtype: {'bfloat16' if torch.cuda.is_bf16_supported() else 'float16'}")
else:
    print("ACTION REQUIRED: Runtime > Change runtime type > GPU, then rerun this cell.")


Python: 3.12.13
PyTorch: 2.11.0+cu128
PyTorch CUDA runtime: 12.8
CUDA usable: True
Host RAM: 12.67 GiB
Tesla T4, 580.82.07, 15360 MiB
GPU: Tesla T4; memory: 14.56 GiB
Mixed precision dtype: bfloat16


## Persist exact provenance

The final cell writes both a timestamped record and `outputs/environment/latest.json`. It captures Python, OS, package versions, CUDA data, RAM, git state for the project and baselines, and the checkpoint destination. This is the provenance artifact that later notebooks should reference when generating results.

**Estimated resources:** under one minute; no material GPU memory. **Expected output:** paths to two JSON files and a concise environment summary.

In [ ]:
# Cell 6 - Write reproducibility metadata and print the final environment summary.
from __future__ import annotations

import importlib.metadata
import json
import os
import platform
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import psutil
import torch

PROJECT_ROOT = Path(os.environ.get("NRC_CAL_PROJECT_ROOT", "/content/NRC_CALIB_CODE")).expanduser()
CHECKPOINT_ROOT = Path(os.environ.get("NRC_CAL_CHECKPOINT_ROOT", str(PROJECT_ROOT / "checkpoints"))).expanduser()
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
ENVIRONMENT_DIR = PROJECT_ROOT / "outputs" / "environment"
ENVIRONMENT_DIR.mkdir(parents=True, exist_ok=True)
DISTRIBUTIONS = (
    "torch", "lightning", "numpy", "scipy", "pandas", "matplotlib", "plotly", "seaborn",
    "scikit-learn", "hydra-core", "tqdm", "einops", "PyYAML", "rich", "jupyterlab",
    "wandb", "umap-learn", "psutil",
)

def command_output(command: list[str], cwd: Path | None = None) -> str | None:
    """Return command output, or None when the executable/command is unavailable."""
    try:
        completed = subprocess.run(command, cwd=cwd, check=True, text=True, capture_output=True)
    except (FileNotFoundError, subprocess.CalledProcessError):
        return None
    return completed.stdout.strip()

def git_metadata(path: Path) -> dict[str, str | None]:
    """Collect git identity without failing for Drive-only directories."""
    return {
        "commit": command_output(["git", "rev-parse", "HEAD"], path),
        "remote": command_output(["git", "config", "--get", "remote.origin.url"], path),
        "status": command_output(["git", "status", "--short"], path),
    }

packages = {}
for distribution in DISTRIBUTIONS:
    try:
        packages[distribution] = importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        packages[distribution] = None
cuda_devices: list[dict[str, Any]] = []
for index in range(torch.cuda.device_count()):
    properties = torch.cuda.get_device_properties(index)
    cuda_devices.append({"index": index, "name": properties.name, "memory_bytes": properties.total_memory})
timestamp = datetime.now(timezone.utc)
record: dict[str, Any] = {
    "created_utc": timestamp.isoformat(),
    "project_root": str(PROJECT_ROOT.resolve()),
    "checkpoint_root": str(CHECKPOINT_ROOT.resolve()),
    "python": {"version": sys.version, "executable": sys.executable},
    "platform": {"system": platform.system(), "release": platform.release(), "machine": platform.machine()},
    "ram_bytes": psutil.virtual_memory().total,
    "pytorch": {"version": torch.__version__, "cuda_runtime": torch.version.cuda, "cuda_available": torch.cuda.is_available()},
    "cuda_devices": cuda_devices,
    "nvidia_smi": command_output(["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"]),
    "packages": packages,
    "project_git": git_metadata(PROJECT_ROOT),
    "upstream_git": {name: git_metadata(PROJECT_ROOT / "external" / name) for name in ("quantile-recalibration-training", "probabilistic-calibration-study")},
}
timestamped_path = ENVIRONMENT_DIR / f"environment_{timestamp.strftime('%Y%m%dT%H%M%SZ')}.json"
latest_path = ENVIRONMENT_DIR / "latest.json"
payload = json.dumps(record, indent=2, sort_keys=True) + "\n"
timestamped_path.write_text(payload, encoding="utf-8")
latest_path.write_text(payload, encoding="utf-8")

print("NRC-Cal environment summary")
print(f"  project root: {record['project_root']}")
print(f"  checkpoint root: {record['checkpoint_root']}")
print(f"  Python: {platform.python_version()}")
print(f"  PyTorch: {torch.__version__}; CUDA runtime: {torch.version.cuda}; usable: {torch.cuda.is_available()}")
print(f"  GPU(s): {', '.join(device['name'] for device in cuda_devices) or 'none'}")
print(f"  RAM: {record['ram_bytes'] / 2**30:.2f} GiB")
print(f"  timestamped record: {timestamped_path}")
print(f"  latest record: {latest_path}")


NRC-Cal environment summary
  project root: /content/drive/MyDrive/NRC_CALIB_CODE
  checkpoint root: /content/drive/MyDrive/NRC_CALIB_CODE/checkpoints
  Python: 3.12.13
  PyTorch: 2.11.0+cu128; CUDA runtime: 12.8; usable: True
  GPU(s): Tesla T4
  RAM: 12.67 GiB
  timestamped record: /content/drive/MyDrive/NRC_CALIB_CODE/outputs/environment/environment_20260803T095525Z.json
  latest record: /content/drive/MyDrive/NRC_CALIB_CODE/outputs/environment/latest.json


## Completion checklist

Proceed only when the final summary reports a GPU-capable CUDA PyTorch build and `outputs/environment/latest.json` exists. If the checkpoint inventory says `INCOMPLETE`, that is an accurate result rather than a silent pass: the next notebook must follow the upstream repository's documented checkpoint acquisition procedure and rerun the inventory.

No datasets, models, features, NRC metrics, calibration maps, or experiment results are created here.